# 00 — 环境配置与验证

**BWV861 音乐形式化分析系统** · Step 0

本 Notebook 完成：
1. 验证所有依赖库是否正确安装
2. 确认 `music21` 能成功解析 BWV861 的 `.mxl` 文件
3. 确认 `music_analysis` 共享包可正常导入
4. 查看 score 的基本结构信息

## 1. 依赖库验证

In [2]:
import sys, os, pickle, json
print(f"Python: {sys.version}")

# 核心科学计算
import numpy as np;          print(f"numpy:     {np.__version__}")
import scipy;                print(f"scipy:     {scipy.__version__}")

# 音乐解析
import music21;              print(f"music21:   {music21.__version__}")

# 序列距离
import editdistance;         print(f"editdistance: installed ✓")
from dtw import dtw;         print(f"dtw-python:   installed ✓")

# 图与可视化
import networkx as nx;       print(f"networkx:  {nx.__version__}")
import plotly;               print(f"plotly:    {plotly.__version__}")
import matplotlib;           print(f"matplotlib:{matplotlib.__version__}")

print("\n✅ 所有依赖库导入成功")

Python: 3.11.15 | packaged by Anaconda, Inc. | (main, Jun 11 2026, 15:12:53) [MSC v.1942 64 bit (AMD64)]
numpy:     2.4.6
scipy:     1.17.1
music21:   10.3.0
editdistance: installed ✓
Importing the dtw module. When using in academic works please cite:
  T. Giorgino. Computing and Visualizing Dynamic Time Warping Alignments in R: The dtw Package.
  J. Stat. Soft., doi:10.18637/jss.v031.i07.

dtw-python:   installed ✓
networkx:  3.6.1
plotly:    6.8.0
matplotlib:3.11.0

✅ 所有依赖库导入成功


## 2. MusicXML 解析测试

使用 `music21.converter.parse()` 加载 BWV861 Prelude in G minor。

In [3]:
from music21 import converter

# 切换到项目根目录
os.chdir(r"e:\大三\Cline Projects\bien_project")

score = converter.parse("BWV861_Prelude_G_minor.mxl")
print(f"Score 类型: {type(score).__name__}")
print(f"声部数 (Parts): {len(score.parts)}")
print()

for i, part in enumerate(score.parts):
    part_id = part.id
    measures = list(part.getElementsByClass("Measure"))
    notes_count = len(part.flatten().getElementsByClass("Note"))
    print(f"  Part {i}: id='{part_id}', 小节数={len(measures)}, 音符/和弦数≈{notes_count}")

print("\n✅ music21 解析 BWV861 成功")

d:\anaconda_envs\bien_music_env\Lib\site-packages\music21\musicxml\xmlToM21.py:5516: MusicXMLWarning: Line <bracket> stop without start
  spannerList = self.xmlDirectionTypeToSpanners(


Score 类型: Score
声部数 (Parts): 2

  Part 0: id='P1-Staff1', 小节数=53, 音符/和弦数≈788
  Part 1: id='P1-Staff2', 小节数=53, 音符/和弦数≈554

✅ music21 解析 BWV861 成功


## 3. Score 结构详情

深入查看第一个声部的节拍、调号和小节结构。

In [6]:
from music21 import meter, key, stream

# 时间签名
ts_list = score.flatten().getElementsByClass(meter.TimeSignature)
print("时间签名:")
for ts in ts_list:
    print(f"  {ts.numerator}/{ts.denominator}  (offset={ts.offset})")

# 调号
key_list = score.flatten().getElementsByClass(key.KeySignature)
print("\n调号:")
for k in key_list:
    # KeySignature uses .sharps (positive=sharps, negative=flats)
    print(f"  sharps={k.sharps}  (offset={k.offset})")

# 第一个声部的前 3 个小节预览
print("\n--- Part 0 前 3 小节预览 ---")
first_part = score.parts[0]
for m in first_part.getElementsByClass(stream.Measure)[:3]:
    print(f"\n  Measure {m.number}:")
    for el in m:
        cls = type(el).__name__
        if hasattr(el, 'pitch'):
            print(f"    {cls}: pitch={el.pitch.nameWithOctave}, dur={el.duration.quarterLength}QL, offset={el.offset}")
        elif hasattr(el, 'value'):
            print(f"    {cls}: value={el.value}")
        else:
            print(f"    {cls}")

时间签名:
  4/4  (offset=0.0)
  4/4  (offset=0.0)
  4/4  (offset=76.0)
  4/4  (offset=76.0)

调号:
  sharps=-2  (offset=0.0)
  sharps=-2  (offset=0.0)

--- Part 0 前 3 小节预览 ---

  Measure 1:
    SystemLayout
    TrebleClef
    KeySignature
    TimeSignature
    Note: pitch=G5, dur=4.0QL, offset=0.0

  Measure 2:
    Note: pitch=G5, dur=0.25QL, offset=0.0
    Note: pitch=D5, dur=0.25QL, offset=0.25
    Note: pitch=E-5, dur=0.125QL, offset=0.5
    Note: pitch=D5, dur=0.125QL, offset=0.625
    Note: pitch=C5, dur=0.25QL, offset=0.75
    Note: pitch=D5, dur=0.25QL, offset=1.0
    Note: pitch=G5, dur=0.25QL, offset=1.25
    Note: pitch=C5, dur=0.125QL, offset=1.5
    Note: pitch=B-4, dur=0.125QL, offset=1.625
    Note: pitch=A4, dur=0.25QL, offset=1.75
    Note: pitch=B-4, dur=0.25QL, offset=2.0
    Note: pitch=G5, dur=0.25QL, offset=2.25
    Note: pitch=C5, dur=0.125QL, offset=2.5
    Note: pitch=B-4, dur=0.125QL, offset=2.625
    Note: pitch=A4, dur=0.25QL, offset=2.75
    Note: pitch=B-4, dur=0

## 4. 共享包导入 + data 目录确认

In [5]:
# 确保 music_analysis 包在 sys.path 中
sys.path.insert(0, os.getcwd())

import music_analysis
from music_analysis import config

print("music_analysis 包导入成功")
print(f"  L_MIN={config.L_MIN}, L_MAX={config.L_MAX}")
print(f"  D_MIN={config.D_MIN}, D_MAX={config.D_MAX}, B_MAX={config.B_MAX}")
print(f"  w_I={config.W_I}, w_R={config.W_R}, w_T={config.W_T}")
print(f"  λ_sym={config.LAMBDA_SYM}, λ_temp={config.LAMBDA_TEMP}")
print()

# 确认 data/ 目录
data_dir = config.DATA_DIR
if os.path.isdir(data_dir):
    print(f"✅ '{data_dir}/' 目录已存在")
else:
    os.makedirs(data_dir)
    print(f"📁 已创建 '{data_dir}/' 目录")

# 列出当前项目文件
print("\n📂 项目文件结构:")
for root, dirs, files in os.walk(".", topdown=True):
    level = root.count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root) or '.'}/")
    sub_indent = "  " * (level + 1)
    for f in files:
        print(f"{sub_indent}{f}")
    if level > 2:
        break

music_analysis 包导入成功
  L_MIN=3, L_MAX=8
  D_MIN=1.0, D_MAX=8.0, B_MAX=2
  w_I=0.5, w_R=0.3, w_T=0.2
  λ_sym=1.0, λ_temp=0.5

✅ 'data/' 目录已存在

📂 项目文件结构:
./
  BWV861_Prelude_G_minor.mxl
  formalization_v1.md
  data/
  music_analysis/
    config.py
    distance.py
    dynamics.py
    events.py
    features.py
    motifs.py
    network.py
    themes.py
    viz.py
    windows.py
    __init__.py
    __pycache__/
      config.cpython-311.pyc
      distance.cpython-311.pyc
      dynamics.cpython-311.pyc
      events.cpython-311.pyc
      features.cpython-311.pyc
      motifs.cpython-311.pyc
      network.cpython-311.pyc
      themes.cpython-311.pyc
      viz.cpython-311.pyc
      windows.cpython-311.pyc
      __init__.cpython-311.pyc
  notebooks/
    00_environment_setup.ipynb


## ✅ 环境配置完成

所有检查通过：
- [x] Python 3.11 + 全部依赖库已就绪
- [x] `music21` 成功解析 `BWV861_Prelude_G_minor.mxl`
- [x] `music_analysis` 共享包可正常导入
- [x] `data/` 目录已确认存在
- [x] 超参数已在 `config.py` 中定义

**下一步** → `01_event_extraction.ipynb`：从 Score 中提取离散音乐事件。<｜end▁of▁thinking｜>现在让我运行 00 notebook 来验证一切正常。

<｜｜DSML｜｜tool_calls>
<｜｜DSML｜｜invoke name="run_notebook_cell">
<｜｜DSML｜｜parameter name="cellId" string="true">#VSC-4dbb12ec